# music practice

1. Agent 분기 처리
2. db sub_graph 처리


In [ ]:
import os,sys
sys.path.append(os.path.abspath(os.path.join(os.path.dirname('utils'), '..')))
from module.utils import * 
from module.prompt import * 
from module.custom_model import *
from module.base_model import *

In [ ]:
start_langsmith('practice_00')

LangSmith 추적을 시작합니다.
[프로젝트명]
practice_00


In [ ]:
from typing import TypedDict, Annotated, List, Literal,Tuple
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import AIMessage,HumanMessage,SystemMessage,ToolMessage
from langgraph.prebuilt import create_react_agent
from langchain_core.tools import tool
from langchain_core.runnables import RunnableLambda, RunnableWithFallbacks


### 일반 함수 선언

In [ ]:
class SubState(TypedDict):
    messages: Annotated[list, add_messages]

In [ ]:
# 오류 처리 함수
def handle_tool_error(state) -> dict:
    # 오류 정보 조회
    error = state.get("error")
    # 도구 정보 조회
    tool_calls = state["messages"][-1].tool_calls
    # ToolMessage 로 래핑 후 반환
    return {
        "messages": [
            ToolMessage(
                content=f"Here is the error: {repr(error)}\n\nPlease fix your mistakes.",
                tool_call_id=tc["id"],
            )
            for tc in tool_calls
        ]
    }

def tool_node_with_fallback(tools:list) -> RunnableWithFallbacks[Any, dict]:
    """
    Create a ToolNode with a fallback to handle errors and surface them to the agent.
    """
    # 오류 발생 시 대체 동작을 정의하여 ToolNode에 추가
    return ToolNode(tools).with_fallbacks(
        [RunnableLambda(handle_tool_error)], exception_key="error"
    )


# 쿼리 실행 부
@tool
def db_query_tool(query: str) -> str:
    """
    Run SQL queries against a database and return results
    Returns an error message if the query is incorrect
    If an error is returned, rewrite the query, check, and retry
    """
    # 쿼리 실행
    db = get_db()
    result = db.run_no_throw(query)

    # 오류: 결과가 없으면 오류 메시지 반환
    if not result:
        return "Error: Query failed. Please rewrite your query and try again."
    # 정상: 쿼리 실행 결과 반환
    return result

# 쿼리 오타 검정 
def get_query_check_node(query):
    prompt = get_prompt_query_check()
    llm = get_gpt().bind_tools([db_query_tool],tool_choice='db_query_tool')
    chain = prompt | llm 
    return  {"messages": [chain.invoke({"messages":query})]}

### 노드 선언

In [ ]:

def get_table_list_node(state: SubState) -> dict[str, list[AIMessage]]:
    llm = get_gemini()
    tools = get_db_tool(llm)
    sql_db_list_tables = next(tool for tool in tools if tool.name == "sql_db_list_tables")
    llm_get_schema = llm.bind_tools([sql_db_list_tables],tool_choice='sql_db_list_tables')
    return SubState({ "messages": [llm_get_schema.invoke(state["messages"])]})

def get_all_table_node(state:SubState):
    llm = get_gemini()
    tools = get_db_tool(llm)
    sql_db_list_tables = next(tool for tool in tools if tool.name == "sql_db_list_tables")
    return tool_node_with_fallback([sql_db_list_tables])

def get_one_table_info_node(state:SubState):
    llm = get_gemini()
    tools = get_db_tool(llm)
    sql_db_schema = next(tool for tool in tools if tool.name == "sql_db_schema")
    llm_with_schema = llm.bind_tools([sql_db_schema],tool_choice='sql_db_schema')
    result = llm_with_schema.invoke(state['messages'])
    return SubState({'messages':[result]})

def get_one_table_schema_node(state:SubState):
    llm = get_gemini()
    tools = get_db_tool(llm)
    sql_db_schema = next(tool for tool in tools if tool.name == "sql_db_schema")
    return tool_node_with_fallback([sql_db_schema])

def get_query_gen_node(state:SubState):
    prompt = get_prompt_query_gen()
    llm =get_gemini()
    query_gen_llm = prompt | llm.bind_tools([get_query_check_node],tool_choice='get_query_check_node')
    # query_gen_llm = prompt | llm
    history = state["messages"]
    query_gen =query_gen_llm.invoke({'placeholder':history})
    return SubState({ "messages": [query_gen]})

def execute_query(state:SubState):
    query = ''
    messages = state["messages"][-1]
    if len(messages.tool_calls) > 0:
        query = messages.tool_calls[0]['args']['query']
    else:
        query = state["messages"][-1].content
    return SubState({'messages':db_query_tool(query)})

def answer_node(state:SubState):
    prompt = get_prompt_query_gen()
    llm =get_gemini()
    query_gen_llm = prompt | llm
    history = state["messages"]
    answer =query_gen_llm.invoke({'placeholder':history})
    return SubState({ "messages": [answer]})

### 분기 수행

In [ ]:
# routing 분기처리 수행 
def routing(state: SubState) -> Literal["get_query_gen_node", "answer_node"]:
    latest_messages:str = state["messages"][-1].content
    if "Error:" in latest_messages:
        return "get_query_gen_node"
    else:
        return "answer_node"

### 랭그래프 선언

In [ ]:
sub_state_graph = StateGraph(SubState)
sub_state_graph.add_node('get_table_list_node',get_table_list_node)
sub_state_graph.add_node('get_all_table_node',get_all_table_node)
sub_state_graph.add_node('get_one_table_info_node',get_one_table_info_node)
sub_state_graph.add_node('get_one_table_schema_node',get_one_table_schema_node)
sub_state_graph.add_node("get_query_gen_node", get_query_gen_node)
sub_state_graph.add_node("execute_query", execute_query)
sub_state_graph.add_node("answer_node", answer_node)

# state_graph.add_node("get_query_check_node", get_query_check_node)



sub_state_graph.add_edge(START,'get_table_list_node')
sub_state_graph.add_edge('get_table_list_node','get_all_table_node')
sub_state_graph.add_edge('get_all_table_node','get_one_table_info_node')
sub_state_graph.add_edge('get_one_table_info_node','get_one_table_schema_node')

sub_state_graph.add_edge('get_one_table_schema_node','get_query_gen_node')

sub_state_graph.add_edge('get_query_gen_node','execute_query')
sub_state_graph.add_conditional_edges(
    source='execute_query',
    path=routing
)

sub_state_graph.add_edge('answer_node',END)

sub_ck = get_check_pointer()
sub_graph = sub_state_graph.compile(checkpointer=sub_ck)

In [ ]:
class State(TypedDict):
    messages: Annotated[list, add_messages]

In [ ]:
def retriever():
    loader = get_pdf_loader()
    splitter = get_text_splitter()
    docs = get_docs(loader,splitter)
    embedding = get_embedding()
    retrieve = get_retriever(docs,embedding)
    retrieve_tool = get_retriever_tool(retrieve)
    return retrieve_tool

def db_toolkit():
    db = SQLDatabase.from_uri("sqlite:///Chinook.db")
    toolkit = SQLDatabaseToolkit(db=db, llm=get_gpt())
    return toolkit.get_tools()



In [ ]:
def start_node(state:State):
    prompt = get_prompt_start_node()   
    llm = get_gpt()
    chain = prompt | llm.with_structured_output(RouteModel) 
    result = chain.invoke({'placeholder':state['messages']})
    response = result.datasource
    return State({'messages':[HumanMessage(content=response)]})



def web_search_agent(state:State):
    prompt = get_prompt_multi_web()
    llm = get_gpt()
    tavily = get_tavily_tool()
    tools = [tavily]
    web_agent = create_react_agent(model = llm,tools = tools,prompt=prompt)
    result = web_agent.invoke({'messages':state['messages']})
    print(result)
    latest_messages = HumanMessage(
        content=result["messages"][-1].content, name="web_search"
    )
    return State({'messages':[latest_messages]})


def pdf_loader_agent(state:State):
    prompt = get_prompt_multi_loader()
    llm = get_gpt()
    tool = retriever()
    tools = [tool]
    web_agent = create_react_agent(model = llm,tools = tools,prompt=prompt)
    result = web_agent.invoke({'messages':state['messages']})
    print(result)
    latest_messages = HumanMessage(
        content=result["messages"][-1].content, name="pdf_loader"
    )
    return State({'messages':[latest_messages]})

def db_agent(state:State):
    prompt = get_prompt_assistant()
    llm = get_gpt()
    chain = prompt | llm
    result = chain.invoke({'messages':state['messages']})
    return State({'messages':result})


def conversation_agent(state:State):
    prompt = get_prompt_assistant()
    llm = get_gpt()
    chain = prompt | llm
    result = chain.invoke({'messages':state['messages']})
    return State({'messages':result})


In [ ]:
def is_agent_decision(state: State)->Literal['web_search_agent','pdf_loader_agent','db_agent','conversation_agent']:
    messages = state["messages"]
    last_message = messages[-1].content
    if last_message =='web_search_agent':
        return 'web_search_agent'
    elif last_message =='pdf_loader_agent':
        return 'pdf_loader_agent'
    elif last_message =='db_agent':
        return 'db_agent'
    else: 
        return 'conversation_agent'

def router(state: State):
    # This is the router
    messages = state["messages"]
    last_message = messages[-1]
    if "FINAL ANSWER" in last_message.content:
        # Any agent decided the work is done
        return END
    return "continue"

In [ ]:
state_graph = StateGraph(State)
state_graph.add_node('start_node',start_node)
state_graph.add_edge(START,'start_node')
state_graph.add_node('web_search_agent',web_search_agent)
state_graph.add_node('pdf_loader_agent',pdf_loader_agent)
state_graph.add_node('db_agent',sub_graph)
state_graph.add_node('conversation_agent',conversation_agent)

state_graph.add_conditional_edges(
    source = "start_node",
    path = is_agent_decision,
)


ck = get_check_pointer()
graph = state_graph.compile(checkpointer=ck)


In [ ]:
# visualize_graph(graph,xray=True)

In [ ]:
config = get_runnable_config(recursion_limit=20,thread_id=get_random_uuid())
inputs = {'messages':['Andrew Adam 직원의 인적정보를 데이터 베이스에서 찾아줘']}
stream_graph(graph,inputs,config)




🔄 Node: start_node 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
{"datasource":"db_agent"}db_agent

/tmp/ipykernel_384304/3923261446.py:44: LangChainDeprecationWarning: The method `BaseTool.__call__` was deprecated in langchain-core 0.1.47 and will be removed in 1.0. Use :meth:`~invoke` instead.
  return SubState({'messages':db_query_tool(query)})



🔄 Node: db_agent 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
Album, Artist, Customer, Employee, Genre, Invoice, InvoiceLine, MediaType, Playlist, PlaylistTrack, Track
CREATE TABLE "Employee" (
	"EmployeeId" INTEGER NOT NULL, 
	"LastName" NVARCHAR(20) NOT NULL, 
	"FirstName" NVARCHAR(20) NOT NULL, 
	"Title" NVARCHAR(30), 
	"ReportsTo" INTEGER, 
	"BirthDate" DATETIME, 
	"HireDate" DATETIME, 
	"Address" NVARCHAR(70), 
	"City" NVARCHAR(40), 
	"State" NVARCHAR(40), 
	"Country" NVARCHAR(40), 
	"PostalCode" NVARCHAR(10), 
	"Phone" NVARCHAR(24), 
	"Fax" NVARCHAR(24), 
	"Email" NVARCHAR(60), 
	PRIMARY KEY ("EmployeeId"), 
	FOREIGN KEY("ReportsTo") REFERENCES "Employee" ("EmployeeId")
)

/*
3 rows from Employee table:
EmployeeId	LastName	FirstName	Title	ReportsTo	BirthDate	HireDate	Address	City	State	Country	PostalCode	Phone	Fax	Email
1	Adams	Andrew	General Manager	None	1962-02-18 00:00:00	2002-08-14 00:00:00	11120 Jasper Ave NW	Edmonton	AB	Canada	T5K 2N1	+1 (780) 428-9482	+1 (780) 428

In [ ]:

snapshot = graph.get_state(config)
snapshot

StateSnapshot(values={'messages': [HumanMessage(content='Andrew Adam 직원의 인적정보를 데이터 베이스에서 찾아줘', additional_kwargs={}, response_metadata={}, id='b6dee5ae-a571-495f-8ec0-3da118deb394'), HumanMessage(content='db_agent', additional_kwargs={}, response_metadata={}, id='0ecf4af2-9721-47ef-9840-eb4b1e1f0205'), AIMessage(content='', additional_kwargs={'function_call': {'name': 'sql_db_list_tables', 'arguments': '{"tool_input": ""}'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': []}, id='run--4576403c-e93d-4a12-ab3f-c71a216cefe1', tool_calls=[{'name': 'sql_db_list_tables', 'args': {'tool_input': ''}, 'id': 'ecc85e83-31b6-4596-a8cd-06da69d801a2', 'type': 'tool_call'}], usage_metadata={'input_tokens': 84, 'output_tokens': 19, 'total_tokens': 103, 'input_token_details': {'cache_read': 0}}), ToolMessage(content='Album, Artist, Customer, Employee, Genre, Invoice, InvoiceLine, MediaType, Playlist, PlaylistTrack, Track', name='sql_db_list_tables'